# E9: Population distribution match

**Question.** Can we draw synthetic plants from a distribution fitted to the E5 procedural parameters and reproduce the population statistics of the real 324-plant set? This is the generative validation: if YES, then PhenoSuite's 3-parameter procedural representation is a usable foundation for synthetic-plant generation, GWAS power simulations, and downstream-pipeline stress tests.

**Setup.** Inputs are the E5 fitted parameters (`outputs/e5_fitted_params.csv`): per-leaf `leaf_length` (m), `leaf_angle` (deg), `droopiness` (unitless), and `distance` (internode spacing, m). 324 plants, 2,479 leaves total. Most plants have 5-9 leaves with usable fits; node 0 is the lowest leaf.

**Two generative models compared:**
1. **Marginal model (A)**: each (parameter, node) is treated as an independent univariate Gaussian. Leaves are sampled independently. Tests whether the marginal distributions are sufficient.
2. **Joint model (B)**: each parameter gets one multivariate Gaussian over the full per-node vector (across nodes for one plant). Preserves within-plant cross-node correlations — a plant that is 'big' at node 0 stays 'big' at node 1.

**Comparison metrics.** Per-node KS test (real vs synthetic for each parameter), per-node mean and std agreement, and three plant-level aggregates (total plant height = sum of distances, mean leaf angle, total leaf area proxy = sum of `leaf_length`).

**What 'match' means here.** We are not testing that synthetic plants are biologically valid — only that *as a population*, they share the same statistics as the real 324-plant SAP subset. Individual-plant realism is downstream of this and is not in scope for E9.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == '01_phyllotaxis' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

OUT = REPO_ROOT / 'experiments' / '01_phyllotaxis' / 'outputs'
FIG = REPO_ROOT / 'experiments' / '01_phyllotaxis' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(20250527)
PARAMS = ['leaf_length', 'leaf_angle', 'droopiness', 'distance']

print('Imports OK.')

## 1. Load the fitted procedural parameters

From E5: each row is one leaf with its fitted procedural parameters. We restrict to leaves with reasonable fit quality (`rms_error_cm <= 90th percentile` consistent with the E5 outlier filter) so the population we are matching is the same one E5 reported on.

In [ ]:
raw = pd.read_csv(OUT / 'e5_fitted_params.csv')

# Same filter as E5: voxel >= 20 + RMS <= 90th percentile
rms_cap = raw['rms_error_cm'].quantile(0.90)
real = raw[(raw['leaf_voxel_count'] >= 20) & (raw['rms_error_cm'] <= rms_cap)].copy()
real = real[['plant_id', 'leaf_index'] + PARAMS].copy()

n_plants = real['plant_id'].nunique()
leaves_per_plant = real.groupby('plant_id').size()
print(f'Filtered population: {len(real):,} leaves across {n_plants} plants')
print(f'  leaves/plant: min={leaves_per_plant.min()}  '
      f'median={int(leaves_per_plant.median())}  max={leaves_per_plant.max()}')
real.head(3)

## 2. Empirical per-node marginals (real population)

Mean +/- std per node for each parameter. These should reproduce the per-node gradients reported in E5 (leaf_angle rising with node, droopiness magnitude decreasing, length following a bell curve).

In [ ]:
by_node = real.groupby('leaf_index').agg(
    n=('plant_id', 'count'),
    leaf_length_mean=('leaf_length', 'mean'),
    leaf_length_std=('leaf_length', 'std'),
    leaf_angle_mean=('leaf_angle', 'mean'),
    leaf_angle_std=('leaf_angle', 'std'),
    droopiness_mean=('droopiness', 'mean'),
    droopiness_std=('droopiness', 'std'),
    distance_mean=('distance', 'mean'),
    distance_std=('distance', 'std'),
).round(3)
by_node

## 3. Model A: independent per-node marginals

For each (parameter, node) we fit an independent Gaussian — just the empirical mean and std. To synthesize a plant of K leaves we draw `K * 4` independent samples. This is the simplest possible generator and serves as the baseline.

In [ ]:
# Fit per-node Gaussians for each parameter
node_dist = {}
MAX_NODE = int(real['leaf_index'].max())
node_counts = real.groupby('leaf_index').size().to_dict()
for p in PARAMS:
    grp = real.groupby('leaf_index')[p].agg(['mean', 'std', 'count'])
    node_dist[p] = grp

# Empirical leaves-per-plant distribution (so synthetic plants have realistic node counts)
lpp = leaves_per_plant.value_counts().sort_index()
leaf_count_pmf = lpp / lpp.sum()
print('Leaves-per-plant empirical PMF (real population):')
print(leaf_count_pmf.round(3).to_dict())

In [ ]:
def sample_population_A(n_plants_synth: int, rng: np.random.Generator) -> pd.DataFrame:
    """Model A: independent per-node Gaussian sampling."""
    rows = []
    leaf_counts = rng.choice(leaf_count_pmf.index.values, size=n_plants_synth,
                              p=leaf_count_pmf.values, replace=True)
    for pid_idx, K in enumerate(leaf_counts):
        plant_id = f'synth_A_{pid_idx:04d}'
        for node in range(int(K)):
            if node > MAX_NODE:
                continue
            row = {'plant_id': plant_id, 'leaf_index': node}
            for p in PARAMS:
                mu = node_dist[p].loc[node, 'mean']
                sd = node_dist[p].loc[node, 'std']
                row[p] = float(rng.normal(mu, sd))
            rows.append(row)
    return pd.DataFrame(rows)

synth_A = sample_population_A(n_plants, RNG)
print(f'Model A synthetic: {len(synth_A):,} leaves across {synth_A["plant_id"].nunique()} plants')

## 4. Model B: per-parameter multivariate Gaussian across nodes

Model A treats nodes as independent, but within a plant the parameter at node $i$ is strongly correlated with the parameter at node $i+1$ (a plant with big leaves at the base usually has big leaves nearby too). Model B fits one multivariate Gaussian per parameter over the per-node vector $(p_0, p_1, \ldots, p_K)$ across plants, then samples that vector jointly. This preserves within-plant smoothness.

**Implementation note.** Because leaf counts vary per plant (5–9 nodes typical), we fit the joint distribution on the lower N_FIT nodes where almost all plants have a value (using pairwise-complete covariance estimation on the smaller subset). For synthesis, we sample the full vector then truncate to a realistic leaf count from the empirical PMF.

In [ ]:
# Use a node count where >= 50% of plants have a leaf, so the covariance estimate is stable
node_coverage = (real.groupby('leaf_index').size() / n_plants).round(3)
N_FIT = int((node_coverage >= 0.5).sum())
print(f'Node coverage (fraction of plants with a leaf at each node):')
print(node_coverage.to_dict())
print(f'\nN_FIT = {N_FIT} (nodes 0..{N_FIT-1}) for the multivariate Gaussian fit.')

# Pivot to wide: one row per plant, columns are (param, node)
wide_real = real.pivot_table(index='plant_id', columns='leaf_index', values=PARAMS, aggfunc='first')
# Restrict to the first N_FIT nodes
joint_params = {}
for p in PARAMS:
    block = wide_real[p].iloc[:, :N_FIT].copy()
    # Need a sufficient set of plants with complete data for these nodes
    block_clean = block.dropna()
    mu = block_clean.mean().values
    cov = np.cov(block_clean.values, rowvar=False)
    # Small ridge to ensure PD
    cov = cov + 1e-6 * np.eye(cov.shape[0])
    joint_params[p] = {'mu': mu, 'cov': cov, 'n_plants': len(block_clean)}
    print(f'  {p}: fit on {len(block_clean)} complete-data plants  '
          f'mu range [{mu.min():.3g}, {mu.max():.3g}]')

In [ ]:
def sample_population_B(n_plants_synth: int, rng: np.random.Generator) -> pd.DataFrame:
    """Model B: per-parameter multivariate-Gaussian sampling over the lower N_FIT nodes.
    
    Higher nodes (>= N_FIT) fall back to Model A's independent per-node sampling.
    """
    leaf_counts = rng.choice(leaf_count_pmf.index.values, size=n_plants_synth,
                              p=leaf_count_pmf.values, replace=True)
    # Vectorized MVN draw per parameter for the lower N_FIT block
    blocks = {}
    for p in PARAMS:
        blocks[p] = rng.multivariate_normal(
            joint_params[p]['mu'], joint_params[p]['cov'], size=n_plants_synth
        )
    rows = []
    for pid_idx, K in enumerate(leaf_counts):
        plant_id = f'synth_B_{pid_idx:04d}'
        for node in range(int(K)):
            if node > MAX_NODE:
                continue
            row = {'plant_id': plant_id, 'leaf_index': node}
            for p in PARAMS:
                if node < N_FIT:
                    row[p] = float(blocks[p][pid_idx, node])
                else:
                    mu = node_dist[p].loc[node, 'mean']
                    sd = node_dist[p].loc[node, 'std']
                    row[p] = float(rng.normal(mu, sd))
            rows.append(row)
    return pd.DataFrame(rows)

synth_B = sample_population_B(n_plants, RNG)
print(f'Model B synthetic: {len(synth_B):,} leaves across {synth_B["plant_id"].nunique()} plants')

## 5. Per-node marginal comparison (KS test)

For each (parameter, node) we run a two-sample Kolmogorov–Smirnov test between the real and synthetic populations. A small KS statistic and large p-value indicates the synthetic marginal matches the real one well.

In [ ]:
def ks_compare(real_df: pd.DataFrame, synth_df: pd.DataFrame, label: str) -> pd.DataFrame:
    rows = []
    for p in PARAMS:
        for node in range(MAX_NODE + 1):
            real_vals = real_df.loc[real_df['leaf_index'] == node, p].dropna().values
            synth_vals = synth_df.loc[synth_df['leaf_index'] == node, p].dropna().values
            if len(real_vals) < 5 or len(synth_vals) < 5:
                continue
            ks = stats.ks_2samp(real_vals, synth_vals)
            rows.append({
                'model': label, 'param': p, 'leaf_index': node,
                'n_real': len(real_vals), 'n_synth': len(synth_vals),
                'KS_stat': float(ks.statistic), 'KS_p': float(ks.pvalue),
                'mean_real': float(np.mean(real_vals)),
                'mean_synth': float(np.mean(synth_vals)),
                'std_real':  float(np.std(real_vals, ddof=1)),
                'std_synth': float(np.std(synth_vals, ddof=1)),
            })
    return pd.DataFrame(rows)

ks_A = ks_compare(real, synth_A, 'A_marginal')
ks_B = ks_compare(real, synth_B, 'B_joint')
ks_all = pd.concat([ks_A, ks_B], ignore_index=True)

# Quick summary: how many (param, node) pairs have KS p >= 0.05 (cannot reject null of same distribution)?
summary_ks = (
    ks_all.assign(pass_05=(ks_all['KS_p'] >= 0.05),
                   pass_01=(ks_all['KS_p'] >= 0.01))
          .groupby('model').agg(
              n_pairs=('KS_p', 'count'),
              pass_p005=('pass_05', 'sum'),
              pass_p001=('pass_01', 'sum'),
              median_KS_stat=('KS_stat', 'median'),
              mean_KS_stat=('KS_stat', 'mean'),
          )
)
summary_ks

In [ ]:
pd.set_option('display.float_format', '{:.3f}'.format)
ks_all.sort_values(['model', 'param', 'leaf_index']).reset_index(drop=True)

## 6. Overlaid histograms: real vs Model A vs Model B

Per-node histograms for one selected leaf (node 1) showing all three populations on the same axes.

In [ ]:
NODES_TO_PLOT = [0, 1, 2, 3]
fig, axes = plt.subplots(len(NODES_TO_PLOT), len(PARAMS), figsize=(15, 12), sharex='col')
for i, node in enumerate(NODES_TO_PLOT):
    for j, p in enumerate(PARAMS):
        ax = axes[i, j]
        rv = real.loc[real['leaf_index'] == node, p].dropna().values
        sa = synth_A.loc[synth_A['leaf_index'] == node, p].dropna().values
        sb = synth_B.loc[synth_B['leaf_index'] == node, p].dropna().values
        if len(rv) == 0:
            ax.set_visible(False); continue
        lo, hi = np.percentile(rv, [1, 99])
        span = hi - lo
        bins = np.linspace(lo - 0.1 * span, hi + 0.1 * span, 30)
        ax.hist(rv, bins=bins, density=True, alpha=0.55, color='#666666',
                label='real (E5 fit)' if i == 0 and j == 0 else None,
                edgecolor='black', linewidth=0.3)
        ax.hist(sa, bins=bins, density=True, alpha=0.45, color='#4477aa',
                label='Model A (marginal)' if i == 0 and j == 0 else None,
                edgecolor='black', linewidth=0.3)
        ax.hist(sb, bins=bins, density=True, alpha=0.45, color='#cc6677',
                label='Model B (joint)' if i == 0 and j == 0 else None,
                edgecolor='black', linewidth=0.3)
        ax.set_xlim(bins.min(), bins.max())
        if j == 0:
            ax.set_ylabel(f'node {node}\ndensity')
        if i == 0:
            ax.set_title(p)
        if i == len(NODES_TO_PLOT) - 1:
            ax.set_xlabel(p)
fig.legend(loc='upper right', bbox_to_anchor=(0.99, 0.99), ncol=3, fontsize=10)
fig.suptitle('E9 per-node marginals: real vs synthetic populations', y=1.005)
fig.tight_layout()
fig.savefig(FIG / 'e9_marginal_histograms.png', dpi=140, bbox_inches='tight')
plt.show()

## 7. Per-node mean +/- std curves

If the synthetic populations match in marginals, their per-node mean and std curves should overlay the real curves.

In [ ]:
def per_node_summary(df, label):
    g = df.groupby('leaf_index').agg(
        **{p: (p, 'mean') for p in PARAMS},
        **{f'{p}_std': (p, 'std') for p in PARAMS},
        n=('plant_id', 'count'),
    )
    g['label'] = label
    return g

sR = per_node_summary(real, 'real')
sA = per_node_summary(synth_A, 'A')
sB = per_node_summary(synth_B, 'B')

fig, axes = plt.subplots(1, len(PARAMS), figsize=(16, 4))
for ax, p in zip(axes, PARAMS):
    nodes = sR.index.values
    ax.errorbar(nodes, sR[p], yerr=sR[f'{p}_std'], fmt='-o', color='black',
                label='real', capsize=2, linewidth=1.4, markersize=5)
    ax.errorbar(nodes + 0.12, sA.reindex(sR.index)[p],
                yerr=sA.reindex(sR.index)[f'{p}_std'], fmt='-s', color='#4477aa',
                label='Model A', capsize=2, linewidth=1.0, markersize=4, alpha=0.85)
    ax.errorbar(nodes - 0.12, sB.reindex(sR.index)[p],
                yerr=sB.reindex(sR.index)[f'{p}_std'], fmt='-^', color='#cc6677',
                label='Model B', capsize=2, linewidth=1.0, markersize=4, alpha=0.85)
    ax.set_xlabel('leaf node')
    ax.set_ylabel(p)
    ax.set_title(p)
    ax.grid(alpha=0.3)
    if ax is axes[0]:
        ax.legend(loc='best', fontsize=9)
fig.suptitle('E9 per-node mean +/- std: real vs synthetic populations', y=1.02)
fig.tight_layout()
fig.savefig(FIG / 'e9_per_node_meanstd.png', dpi=140, bbox_inches='tight')
plt.show()

## 8. Plant-level aggregates

Three population-level summaries:
- **total plant height** = sum of `distance` over leaves (cumulative internode spacing)
- **mean leaf angle** per plant (averaged across nodes)
- **total leaf-length proxy** = sum of `leaf_length` over leaves

If Model B preserves within-plant correlations correctly, its plant-aggregate distributions should match the real population more closely than Model A (which can produce unrealistic 'mixed' plants by sampling independently per node).

In [ ]:
def plant_aggs(df):
    g = df.groupby('plant_id').agg(
        total_height=('distance', 'sum'),
        mean_leaf_angle=('leaf_angle', 'mean'),
        total_leaf_length=('leaf_length', 'sum'),
        mean_droopiness=('droopiness', 'mean'),
        n_leaves=('leaf_index', 'count'),
    )
    return g

agg_real = plant_aggs(real)
agg_A    = plant_aggs(synth_A)
agg_B    = plant_aggs(synth_B)

AGG_COLS = ['total_height', 'mean_leaf_angle', 'total_leaf_length', 'mean_droopiness']
agg_rows = []
for col in AGG_COLS:
    for label, dfagg in [('A', agg_A), ('B', agg_B)]:
        ks = stats.ks_2samp(agg_real[col].dropna(), dfagg[col].dropna())
        agg_rows.append({
            'aggregate': col, 'model': label,
            'mean_real': agg_real[col].mean(),
            'mean_synth': dfagg[col].mean(),
            'std_real': agg_real[col].std(),
            'std_synth': dfagg[col].std(),
            'KS_stat': float(ks.statistic),
            'KS_p':    float(ks.pvalue),
        })
agg_compare = pd.DataFrame(agg_rows)
agg_compare

In [ ]:
fig, axes = plt.subplots(1, len(AGG_COLS), figsize=(18, 4.2))
for ax, col in zip(axes, AGG_COLS):
    rv = agg_real[col].dropna().values
    sa = agg_A[col].dropna().values
    sb = agg_B[col].dropna().values
    lo, hi = np.percentile(np.concatenate([rv, sa, sb]), [1, 99])
    span = hi - lo
    bins = np.linspace(lo - 0.05 * span, hi + 0.05 * span, 30)
    ax.hist(rv, bins=bins, density=True, alpha=0.6, color='#666666',
            label='real', edgecolor='black', linewidth=0.3)
    ax.hist(sa, bins=bins, density=True, alpha=0.5, color='#4477aa',
            label='Model A', edgecolor='black', linewidth=0.3)
    ax.hist(sb, bins=bins, density=True, alpha=0.5, color='#cc6677',
            label='Model B', edgecolor='black', linewidth=0.3)
    ax.set_xlabel(col)
    ax.set_ylabel('density')
    ax.set_title(col)
    ax.grid(alpha=0.3)
axes[0].legend(loc='best', fontsize=9)
fig.suptitle('E9 plant-level aggregate distributions', y=1.02)
fig.tight_layout()
fig.savefig(FIG / 'e9_plant_aggregates.png', dpi=140, bbox_inches='tight')
plt.show()

## 9. Within-plant cross-node correlations

The key place Model A should fail and Model B should succeed: a plant's `leaf_length` at node 1 is strongly correlated with its `leaf_length` at node 2, because leaf size is partly genetic. If our generator breaks that correlation, synthetic plants look 'mixed' even though their marginals are correct.

In [ ]:
def cross_node_corr(df, p, max_node):
    """Pearson r between (param@node i, param@node i+1) for the same plant, for i=0..max_node-1."""
    wide = df.pivot_table(index='plant_id', columns='leaf_index', values=p, aggfunc='first')
    out = []
    for i in range(max_node):
        x = wide[i].values if i in wide.columns else None
        y = wide[i+1].values if i+1 in wide.columns else None
        if x is None or y is None:
            out.append(np.nan); continue
        mask = ~(np.isnan(x) | np.isnan(y))
        if mask.sum() < 5:
            out.append(np.nan); continue
        r = float(np.corrcoef(x[mask], y[mask])[0, 1])
        out.append(r)
    return out

MAX_CORR_NODE = min(N_FIT, 6)
corr_rows = []
for p in PARAMS:
    cR = cross_node_corr(real,    p, MAX_CORR_NODE)
    cA = cross_node_corr(synth_A, p, MAX_CORR_NODE)
    cB = cross_node_corr(synth_B, p, MAX_CORR_NODE)
    for i, (r, a, b) in enumerate(zip(cR, cA, cB)):
        corr_rows.append({'param': p, 'pair': f'{i}_to_{i+1}',
                          'r_real': r, 'r_A': a, 'r_B': b})
corr_compare = pd.DataFrame(corr_rows)
corr_compare

In [ ]:
fig, axes = plt.subplots(1, len(PARAMS), figsize=(16, 4))
for ax, p in zip(axes, PARAMS):
    sub = corr_compare[corr_compare['param'] == p]
    x = np.arange(len(sub))
    width = 0.27
    ax.bar(x - width, sub['r_real'], width, color='#666666', label='real', edgecolor='black', linewidth=0.5)
    ax.bar(x,         sub['r_A'],    width, color='#4477aa', label='Model A', edgecolor='black', linewidth=0.5)
    ax.bar(x + width, sub['r_B'],    width, color='#cc6677', label='Model B', edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(sub['pair'].tolist(), rotation=45)
    ax.set_ylabel('Pearson r between nodes i and i+1')
    ax.set_title(p)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.grid(axis='y', alpha=0.3)
    if ax is axes[0]:
        ax.legend(loc='best', fontsize=9)
fig.suptitle('E9 within-plant cross-node correlations (Model A breaks them, Model B preserves)', y=1.02)
fig.tight_layout()
fig.savefig(FIG / 'e9_cross_node_correlations.png', dpi=140, bbox_inches='tight')
plt.show()

## 10. Save outputs

In [ ]:
# Per-node marginal Gaussian parameters (Model A spec)
marginal_params = []
for p in PARAMS:
    for node in node_dist[p].index:
        marginal_params.append({
            'param': p, 'leaf_index': int(node),
            'mean':  float(node_dist[p].loc[node, 'mean']),
            'std':   float(node_dist[p].loc[node, 'std']),
            'n':     int(node_dist[p].loc[node, 'count']),
        })
pd.DataFrame(marginal_params).to_csv(OUT / 'e9_marginal_params.csv', index=False)

# Joint Gaussian means + covariance matrices (Model B spec)
joint_records = []
for p in PARAMS:
    mu = joint_params[p]['mu']; cov = joint_params[p]['cov']
    for i in range(len(mu)):
        rec = {'param': p, 'node_i': i, 'mean_i': float(mu[i])}
        for j in range(len(mu)):
            rec[f'cov_with_node_{j}'] = float(cov[i, j])
        joint_records.append(rec)
pd.DataFrame(joint_records).to_csv(OUT / 'e9_joint_params.csv', index=False)

# Synthetic populations themselves
synth_A.to_csv(OUT / 'e9_synthetic_modelA.csv', index=False)
synth_B.to_csv(OUT / 'e9_synthetic_modelB.csv', index=False)

# KS tables (per-node + aggregate)
ks_all.to_csv(OUT / 'e9_ks_per_node.csv', index=False)
agg_compare.to_csv(OUT / 'e9_aggregate_compare.csv', index=False)
corr_compare.to_csv(OUT / 'e9_cross_node_correlations.csv', index=False)
summary_ks.to_csv(OUT / 'e9_summary.csv')

for f in ['e9_marginal_params.csv','e9_joint_params.csv',
          'e9_synthetic_modelA.csv','e9_synthetic_modelB.csv',
          'e9_ks_per_node.csv','e9_aggregate_compare.csv',
          'e9_cross_node_correlations.csv','e9_summary.csv']:
    print(' wrote', OUT / f)
for f in ['e9_marginal_histograms.png','e9_per_node_meanstd.png',
          'e9_plant_aggregates.png','e9_cross_node_correlations.png']:
    print(' wrote', FIG / f)

## Discussion

**What this experiment validates.** That a small parametric model fitted to E5's per-leaf procedural parameters can reproduce the population statistics of the real 324-plant set. If yes, PhenoSuite's 3-parameter descriptor format is a viable target for synthetic-plant generation and downstream tasks like GWAS-power simulation and pipeline stress tests.

**Reading the tables.** Two ways the synthetic populations can fail:
1. **Marginal mismatch** — the wrong per-node mean/std for one or more parameters. Caught by the KS p-values in Section 5 and the mean +/- std curves in Section 7.
2. **Joint structure broken** — within-plant cross-node correlations destroyed. Caught by Section 9 and reflected in plant-level aggregate spread (Section 8). Model A is expected to fail here because it samples nodes independently; Model B is designed to preserve these correlations.

**Expected pattern.**
- Both models should pass on per-node marginals (that's literally what they're fit to).
- Only Model B should reproduce the within-plant correlations.
- Plant-level aggregates that are *sums* over the plant (`total_height`, `total_leaf_length`) will be wider under Model A because uncorrelated noise compounds; Model B should track real spread.

**Caveats.**
- The Gaussian assumption is convenient but may fit some parameters poorly (especially `droopiness`, which has a heavier left tail). A future iteration could swap in lognormal for length or a mixture for droopiness.
- We only fit the joint over the lower N_FIT nodes (typically ~6–7) where coverage is good. Upper nodes fall back to Model A's per-node sampling. This is fine for population statistics but limits the realism of synthetic plants with many leaves.
- We have not validated synthetic plants by running them through PhenoSuite's forward model and comparing the resulting trait distributions to the real ones. That is a natural follow-up but requires meaningful PhenoSuite forward-model runtime; it is out of scope for E9 as currently scoped.

**Outputs.** Distribution-fit parameters (`e9_marginal_params.csv`, `e9_joint_params.csv`) and the synthetic populations themselves (`e9_synthetic_modelA.csv`, `e9_synthetic_modelB.csv`) are saved for downstream use. Per-node KS results in `e9_ks_per_node.csv`; plant-aggregate comparison in `e9_aggregate_compare.csv`; cross-node correlation comparison in `e9_cross_node_correlations.csv`.